# Recurrent Neural Networks (RNN)

A self-contained refresher on the **recurrent neural network** — the foundational architecture for sequence modeling: a network with a *loop* that carries a hidden **state** from one time step to the next, so the same weights process a sequence of arbitrary length one element at a time.

**Domain:** Architectures  ·  **runnable:** yes (a from-scratch NumPy RNN + BPTT, plus an optional PyTorch path)

## 1. What & Why

A **recurrent neural network** processes a sequence \(x_1, x_2, \dots, x_T\) one element at a time, maintaining a **hidden state** \(h_t\) that summarizes everything seen so far. At each step it combines the new input with the previous state:

$$h_t = \tanh(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b_h), \qquad y_t = W_{hy}\,h_t + b_y$$

The crucial detail: **the same weight matrices are reused at every step** (parameter sharing across time), and the loop edge \(h_{t-1} \to h_t\) is what makes the network *recurrent*.

**The problem it solves.** A plain feed-forward net (or an MLP/CNN) takes a fixed-size input and has no notion of order or memory. But language, audio, time series, and sensor streams are **variable-length and order-dependent** — "dog bites man" ≠ "man bites dog." RNNs handle this natively: they accept sequences of *any* length with a *fixed* number of parameters, and their hidden state acts as a running memory of context.

**Reach for an RNN when:**
- Inputs/outputs are **sequences** and **order matters** (text, speech, sensor/IoT streams, financial ticks).
- You need to handle **variable-length** sequences with a fixed parameter budget.
- You want an **online / streaming** model that updates state token-by-token (RNN inference is O(1) memory per step — attractive on edge devices).
- The sequences are **short-to-moderate** and dependencies are mostly local.

**Skip it when:** dependencies span long ranges (vanilla RNNs forget — reach for [[lstm]]/GRU, or [[transformer]]s), you need massively parallel training over long sequences (RNNs are inherently sequential; Transformers parallelize across positions), or the data isn't sequential at all (use an [[cnn]] / MLP). For most modern NLP, attention-based models have displaced RNNs — but the RNN is still the mental foundation for everything that followed, and remains competitive for small, latency-sensitive, streaming workloads.

## 2. Mental Model

> **An RNN is a single small network in a `for` loop, passing a note to itself.** Each step it reads the next input *and* the note it wrote last step, then writes a new note. That note — the hidden state — is the only memory it has.

The standard way to reason about an RNN is to **unroll** the loop across time into a deep feed-forward network that happens to share weights at every layer:

```
   x1        x2        x3        x4          (inputs, one per time step)
   │         │         │         │
   ▼         ▼         ▼         ▼
 ┌────┐    ┌────┐    ┌────┐    ┌────┐
 │RNN │─h1─▶│RNN │─h2─▶│RNN │─h3─▶│RNN │ ...   (SAME weights W reused each step)
 └────┘    └────┘    └────┘    └────┘
   │         │         │         │
   ▼         ▼         ▼         ▼
   y1        y2        y3        y4          (outputs, optional per step)
```

The horizontal arrows (`h1 → h2 → h3`) are the recurrence — the channel through which information flows forward in time. Training runs ordinary backprop on this unrolled graph; because the graph is as deep as the sequence is long, the algorithm gets its own name: **backpropagation through time (BPTT)**. And because the *same* \(W_{hh}\) is multiplied in at every step, the gradient that flows back across many steps is essentially \(W_{hh}\) raised to a power — which either shrinks to zero (**vanishing**) or blows up (**exploding**). That single fact explains most of an RNN's strengths and nearly all of its pain.

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **Hidden state \(h_t\)** | A fixed-size vector that is the network's entire memory of the sequence so far. Initialized to zeros, updated every step. |
| **Recurrence relation** | \(h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)\). The \(W_{hh}h_{t-1}\) term is the loop. |
| **Parameter sharing** | The *same* weights are applied at every time step. This is why an RNN handles any length with fixed parameters (the temporal analogue of a CNN sharing filters across space). |
| **BPTT** | *Backpropagation Through Time* — backprop run over the unrolled graph. Gradients accumulate across all steps a weight participated in. |
| **Truncated BPTT** | Cap how many steps back you propagate gradients (e.g. 35) to bound memory/compute on long sequences. |
| **Vanishing / exploding gradients** | Repeated multiplication by \(W_{hh}\) and \(\tanh'\) shrinks gradients toward 0 (can't learn long-range deps) or grows them without bound (training diverges). |
| **Gradient clipping** | Rescale the gradient if its norm exceeds a threshold — the standard cure for *exploding* gradients. |
| **Gating ([[lstm]]/GRU)** | Add learned gates + an additive cell state so gradients flow without repeated multiplicative shrinkage — the cure for *vanishing* gradients. |
| **Teacher forcing** | In sequence generation, feed the *ground-truth* previous token (not the model's own prediction) as input during training. Speeds convergence; causes train/inference mismatch ("exposure bias"). |
| **Sequence I/O shapes** | one-to-many (captioning), many-to-one (sentiment), many-to-many aligned (tagging), many-to-many shifted (translation — see [[encoder-decoder]]). |
| **Bidirectional RNN** | Run one RNN forward and one backward over the sequence and concatenate states — gives each step both past and future context (only for non-streaming tasks). |

## 4. Setup

The runnable examples below use **NumPy only** — we implement a vanilla RNN cell and full BPTT from scratch, because *seeing the loop and the gradient* is the whole point of the refresher. No GPU, no downloads.

```bash
pip install numpy            # required, the from-scratch examples
pip install torch            # optional, only for the final nn.RNN call-shape cell
```

The last code cell shows the idiomatic **PyTorch `nn.RNN`** equivalent and is gated behind an `os.getenv("RUN_TORCH")` check, so the notebook runs top-to-bottom with just NumPy installed.

In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__, "— from-scratch RNN, CPU-only, no downloads")

numpy 2.5.0 — from-scratch RNN, CPU-only, no downloads


## 5. Worked Examples

### Example 1 — A vanilla RNN cell: the forward pass over a sequence

The entire RNN is the recurrence \(h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)\) applied in a loop. Below we feed a 5-step sequence through a randomly-initialized cell and watch the hidden state evolve. Note how \(h_t\) depends on *both* the current input and the previous state — that dependency is the network's memory.

In [2]:
def rnn_cell(x_t, h_prev, Wxh, Whh, bh):
    """One RNN step: combine current input x_t with previous hidden state h_prev."""
    return np.tanh(Wxh @ x_t + Whh @ h_prev + bh)

rng = np.random.default_rng(0)
H, D = 4, 3                                  # hidden size, input size
Wxh = rng.normal(0, 0.5, (H, D))
Whh = rng.normal(0, 0.5, (H, H))
bh  = np.zeros(H)

seq = rng.normal(0, 1, (5, D))               # a length-5 sequence of 3-dim inputs
h = np.zeros(H)                              # state starts empty
print("hidden-state trajectory (memory accumulating step by step):")
for t, x in enumerate(seq):
    h = rnn_cell(x, h, Wxh, Whh, bh)
    print(f"  t={t}  h = {h}")
print("\nfinal state summarizes the whole sequence:", h)

hidden-state trajectory (memory accumulating step by step):
  t=0  h = [-0.351 -0.259  0.16   0.197]
  t=1  h = [ 0.41   0.382 -0.519 -0.127]
  t=2  h = [-0.352 -0.498  0.727  0.192]
  t=3  h = [ 0.344  0.444 -0.644 -0.587]
  t=4  h = [ 0.397 -0.681 -0.183  0.777]

final state summarizes the whole sequence: [ 0.397 -0.681 -0.183  0.777]


### Example 2 — Train a tiny RNN with BPTT to predict the next value of a sine wave

Now the real thing: a learnable RNN trained end-to-end with **backpropagation through time**. The task is next-step prediction — given the value of a sine wave at step \(t\), predict step \(t{+}1\). The network must use its hidden state to track *where in the cycle* it is. We implement the forward pass, the full BPTT backward pass (note the `dh_next` term carrying gradient backward across time), **gradient clipping** to keep exploding gradients in check, and plain SGD.

In [3]:
def make_batch(T, n, rng):
    """n sine-wave windows of length T; target = input shifted one step ahead."""
    xs, ys = [], []
    for _ in range(n):
        phase = rng.uniform(0, 2 * np.pi)
        t = np.linspace(0, 3 * np.pi, T + 1) + phase
        s = np.sin(t)
        xs.append(s[:-1, None])      # x_t
        ys.append(s[1:,  None])      # x_{t+1}
    return np.array(xs), np.array(ys)

rng = np.random.default_rng(1)
H, T, lr = 16, 20, 0.01
Wxh = rng.normal(0, 0.1, (H, 1)); Whh = rng.normal(0, 0.1, (H, H)); Why = rng.normal(0, 0.1, (1, H))
bh  = np.zeros(H);                by  = np.zeros(1)

def forward(xseq):
    hs, ys = [np.zeros(H)], []
    for x in xseq:
        h = np.tanh(Wxh @ x + Whh @ hs[-1] + bh)
        hs.append(h)
        ys.append(Why @ h + by)
    return hs, ys

for epoch in range(400):
    X, Yt = make_batch(T, 8, rng)
    gWxh = np.zeros_like(Wxh); gWhh = np.zeros_like(Whh); gWhy = np.zeros_like(Why)
    gbh  = np.zeros_like(bh);  gby  = np.zeros_like(by);  loss = 0.0
    for xseq, yseq in zip(X, Yt):
        hs, ys = forward(xseq)
        dh_next = np.zeros(H)
        for t in reversed(range(T)):                 # backprop THROUGH TIME
            dy = ys[t] - yseq[t]
            loss += 0.5 * float((dy ** 2).sum())
            gWhy += np.outer(dy, hs[t + 1]); gby += dy
            dh   = Why.T @ dy + dh_next              # gradient from output + from the future
            draw = (1 - hs[t + 1] ** 2) * dh         # through tanh
            gWxh += np.outer(draw, xseq[t]); gWhh += np.outer(draw, hs[t]); gbh += draw
            dh_next = Whh.T @ draw                   # pass gradient one step further back
    for g in (gWxh, gWhh, gWhy, gbh, gby):
        np.clip(g, -5, 5, out=g)                     # gradient clipping
    nb = len(X)
    Wxh -= lr * gWxh / nb; Whh -= lr * gWhh / nb; Why -= lr * gWhy / nb
    bh  -= lr * gbh  / nb; by  -= lr * gby  / nb
    if epoch % 100 == 0 or epoch == 399:
        print(f"epoch {epoch:3d}   mean loss / step = {loss / (nb * T):.4f}")

X, Yt = make_batch(T, 1, rng)                        # one fresh test window
_, ys = forward(X[0])
pred = np.array([y.item() for y in ys]); true = Yt[0, :, 0]
print("\npredicted next values:", pred[:6])
print("ground-truth         :", true[:6])
print("final-step abs error :", round(abs(pred[-1] - true[-1]), 4))

epoch   0   mean loss / step = 0.2267
epoch 100   mean loss / step = 0.0074
epoch 200   mean loss / step = 0.0065
epoch 300   mean loss / step = 0.0067
epoch 399   mean loss / step = 0.0075

predicted next values: [-1.037 -0.605  0.016  0.541  0.881  0.984]
ground-truth         : [-0.71  -0.313  0.152  0.584  0.889  1.   ]
final-step abs error : 0.0124


### Example 3 — Why long-range learning is hard: watch the gradient vanish

The hardest part of training RNNs is the gradient that flows backward across many steps. Each step back multiplies by \(W_{hh}^{\top}\) and by \(\tanh' = 1 - h^2 \le 1\). If the recurrent weights are small (spectral radius < 1), that product shrinks geometrically — the gradient from step 30 barely reaches step 0, so the network simply can't learn dependencies that long. Larger weights cause the opposite: explosion. This single demo is *the* reason [[lstm]]/GRU gating exists.

In [4]:
def grad_norm_over_time(scale, steps=30, H=20):
    """Propagate a gradient backward through `steps` identical RNN steps; track its norm."""
    W = np.eye(H) * scale            # recurrent weight with spectral radius = scale
    h = np.ones(H) * 0.3
    g = np.ones(H)                   # gradient arriving at the last step
    norms = []
    for _ in range(steps):
        g = W.T @ ((1 - np.tanh(h) ** 2) * g)
        norms.append(np.linalg.norm(g))
    return norms

for scale, label in [(0.5, "small weights -> VANISH"), (1.5, "large weights -> EXPLODE")]:
    n = grad_norm_over_time(scale)
    print(f"{label:28s}  step0={n[0]:.2e}  step10={n[10]:.2e}  step29={n[29]:.2e}")

small weights -> VANISH       step0=2.05e+00  step10=8.23e-04  step29=2.91e-10
large weights -> EXPLODE      step0=6.14e+00  step10=1.46e+02  step29=6.00e+04


### Example 4 — The idiomatic PyTorch equivalent (`nn.RNN`), gated so the notebook always runs

In practice you'd never hand-roll BPTT — `torch.nn.RNN` does the forward pass and autograd handles the backward. This cell shows the real call shape and only executes if you set `RUN_TORCH=1` and have PyTorch installed; otherwise it prints the snippet so the notebook still runs end-to-end with NumPy alone.

In [5]:
import os

TORCH_SNIPPET = """
import torch, torch.nn as nn

rnn = nn.RNN(input_size=1, hidden_size=16, num_layers=1, batch_first=True)
head = nn.Linear(16, 1)

x = torch.randn(8, 20, 1)          # (batch, seq_len, features)
h0 = torch.zeros(1, 8, 16)         # (num_layers, batch, hidden)

out, hN = rnn(x, h0)               # out: (8, 20, 16) per-step states; hN: final state
y = head(out)                      # (8, 20, 1) per-step predictions

loss = ((y[:, :-1] - x[:, 1:]) ** 2).mean()   # next-step prediction loss
loss.backward()                    # autograd does BPTT for you
# torch.nn.utils.clip_grad_norm_(rnn.parameters(), 5.0)   # gradient clipping
print(out.shape, hN.shape, float(loss))
"""

if os.getenv("RUN_TORCH") == "1":
    try:
        exec(TORCH_SNIPPET)
    except ImportError:
        print("PyTorch not installed — run `pip install torch` first.")
else:
    print("[skipped] set RUN_TORCH=1 (and `pip install torch`) to execute.")
    print("Call shape for reference:")
    print(TORCH_SNIPPET)

[skipped] set RUN_TORCH=1 (and `pip install torch`) to execute.
Call shape for reference:

import torch, torch.nn as nn

rnn = nn.RNN(input_size=1, hidden_size=16, num_layers=1, batch_first=True)
head = nn.Linear(16, 1)

x = torch.randn(8, 20, 1)          # (batch, seq_len, features)
h0 = torch.zeros(1, 8, 16)         # (num_layers, batch, hidden)

out, hN = rnn(x, h0)               # out: (8, 20, 16) per-step states; hN: final state
y = head(out)                      # (8, 20, 1) per-step predictions

loss = ((y[:, :-1] - x[:, 1:]) ** 2).mean()   # next-step prediction loss
loss.backward()                    # autograd does BPTT for you
# torch.nn.utils.clip_grad_norm_(rnn.parameters(), 5.0)   # gradient clipping
print(out.shape, hN.shape, float(loss))



## 6. Gotchas & Pitfalls

- **Vanishing gradients are the headline limitation.** A vanilla RNN struggles to learn dependencies more than ~10–20 steps apart (Example 3). If your task needs long-range memory, don't fight it — use [[lstm]] or a GRU, whose additive cell state and gates let gradients flow undiminished. This is *the* reason vanilla RNNs are rarely the final answer.
- **Exploding gradients diverge training silently-then-suddenly.** The fix is cheap and near-universal: **clip the global gradient norm** (e.g. to 1–5) every step. Always do this when training any RNN/LSTM/GRU.
- **Always initialize the hidden state, and reset it across independent sequences.** Carrying state from one unrelated sequence into the next (a classic batching bug) leaks information and corrupts training. Reset `h0 = 0` per sequence (or detach it for truncated BPTT).
- **RNNs are inherently sequential — they don't parallelize across time.** You cannot compute \(h_t\) before \(h_{t-1}\). This makes them slow to train on long sequences compared to [[transformer]]s (which process all positions in parallel) and is a big reason attention models won at scale.
- **Teacher forcing creates exposure bias.** Training on ground-truth previous tokens but generating from the model's *own* (possibly wrong) tokens at inference causes a distribution mismatch — small errors compound. Mitigate with scheduled sampling or by occasionally feeding the model's own predictions during training.
- **Mind the tensor layout.** PyTorch `nn.RNN` defaults to `(seq_len, batch, features)`; pass `batch_first=True` for the more intuitive `(batch, seq_len, features)`. Getting this wrong produces silent garbage, not an error.
- **`tanh` saturates.** For inputs that push pre-activations large, \(\tanh'\to 0\) and the unit stops learning. Keep inputs normalized; this also worsens vanishing gradients.
- **Truncate BPTT on long sequences.** Backpropagating through thousands of steps is memory-prohibitive and gradient-unstable. Cap the backward window (e.g. 35 steps) and detach the carried state between windows.

## 7. When to Use vs Alternatives

| Option | Sweet spot | Trade-off vs a vanilla RNN |
|--------|-----------|-----------------------------|
| **Vanilla RNN** | Short sequences, streaming/online inference, tiny latency-sensitive models, teaching the concept | Simplest and cheapest, O(1) memory per step — but **forgets long-range context** and trains unstably. |
| **[[lstm]] / GRU** | The default RNN for real work: medium-to-long sequences (speech, time series, older NLP) | Gating fixes vanishing gradients and captures long-range deps; ~3–4× the parameters and compute. GRU is a lighter LSTM. |
| **[[transformer]] / attention** | Modern NLP, long-context, anything you can train at scale | Parallelizes across positions and models arbitrary-distance dependencies directly — now dominant. Costs O(T²) attention memory and needs more data; weak for true streaming. |
| **[[mamba-ssm]] (state-space models)** | Long sequences where you want RNN-like O(T) inference *and* parallel training | Linear-time, long-range, parallelizable — a modern "RNN done right," but newer and less battle-tested tooling. |
| **1-D [[cnn]] / TCN** | Sequences with mainly *local* patterns; fast parallel training | Fixed receptive field (no unbounded memory) but trains fast and parallelizes; often a strong, simple baseline. |
| **Plain MLP / feature engineering** | Fixed-size or weakly-ordered inputs | No sequence modeling at all — but if order barely matters, far simpler and faster. |

**Rule of thumb:** for new work, default to a **Transformer** (or an LSTM/GRU if data or latency is tight, or [[mamba-ssm]] for very long sequences). Reach for a **vanilla RNN** when sequences are short, the model must be tiny, or you're streaming token-by-token on constrained hardware — and reach for it *always* when you're trying to understand how any of the others work.

## 8. Resources

- **Andrej Karpathy — "The Unreasonable Effectiveness of Recurrent Neural Networks"** (the essay + `min-char-rnn.py` that this notebook's training loop echoes): <https://karpathy.github.io/2015/05/21/rnn-effectiveness/>
- **Christopher Olah — "Understanding LSTM Networks"** (the canonical visual explainer; starts from the vanilla RNN): <https://colah.github.io/posts/2015-08-Understanding-LSTMs/>
- **Goodfellow, Bengio & Courville — *Deep Learning*, Ch. 10 "Sequence Modeling: Recurrent and Recursive Nets"**: <https://www.deeplearningbook.org/contents/rnn.html>
- **On the difficulty of training RNNs** — Pascanu, Mikolov & Bengio (the vanishing/exploding-gradient and gradient-clipping paper): <https://arxiv.org/abs/1211.5063>
- **PyTorch `nn.RNN` documentation** (shapes, args, `batch_first`): <https://pytorch.org/docs/stable/generated/torch.nn.RNN.html>
- **CS231n — Recurrent Neural Networks lecture notes**: <https://cs231n.github.io/> (see the RNN/LSTM lecture)